# 09 — Monitoring simulation and Databricks handoff

**Objectives**

- Detect schema, missingness, numeric, and categorical input change.
- Separate data/score drift from delayed-label performance evidence.
- Map every local artifact to a governed Databricks counterpart.

**Prerequisite:** lesson 08 registered the adopted artifact.


In [ ]:
import pandas as pd

from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.inference import load_champion
from aai_local_classification.learning import state_exists
from aai_local_classification.monitoring import compare_batches, shifted_batch
from aai_local_classification.tracking import local_paths
from aai_local_classification.workflow import (
    ensure_prepared,
    promote_if_approved,
    run_candidate_selection,
    run_frozen_test_gate,
)
from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings

settings = load_settings()
root = study_root()
print(f"Course state: {root}")
print(f"Experiment: {settings.experiment_name}")


In [ ]:
if not state_exists("promotion.json"):
    selected = run_candidate_selection(settings, root)
    decision = run_frozen_test_gate(settings, root, selected)
    promote_if_approved(settings, decision, root, selected)
paths = local_paths(root)
ensure_prepared(settings, root)
reference = load_split(settings, SplitName.VALIDATION, paths.data_root)
current = shifted_batch(reference, settings.random_seed + 1)
report = compare_batches(reference, current, settings)
pd.Series(
    {
        "maximum_numeric_psi": report.maximum_numeric_psi,
        "maximum_categorical_total_variation": report.maximum_categorical_total_variation,
        "largest_missing_rate_delta": max(
            abs(v) for v in report.missing_rate_delta.values()
        ),
    }
).to_frame("diagnostic")


In [ ]:
pd.DataFrame(
    {
        "numeric_psi": pd.Series(report.numeric_psi),
        "missing_rate_delta": pd.Series(report.missing_rate_delta),
    }
).sort_values("numeric_psi", ascending=False)


In [ ]:
predictor = load_champion(settings, root)
reference_scores = predictor.predict(reference, settings)
current_scores = predictor.predict(current, settings)
pd.Series(
    {
        "reference_predicted_positive_rate": reference_scores.churn_prediction.mean(),
        "current_predicted_positive_rate": current_scores.churn_prediction.mean(),
        "reference_mean_score": reference_scores.churn_probability.mean(),
        "current_mean_score": current_scores.churn_probability.mean(),
    }
).to_frame("value")


The current batch is explicitly simulated. PSI and total variation are
diagnostics with context-dependent thresholds, not universal tests. A shifted
score or input distribution warrants investigation; it does not prove recall or
calibration changed. Those require correctly joined delayed outcomes.

## Local to governed platform

| Here | Databricks |
|---|---|
| CSV + digest | versioned Unity Catalog Delta table |
| SQLite MLflow | hosted MLflow tracking |
| local model name | `<catalog>.<schema>.<model>` |
| local `champion` | Models in Unity Catalog alias |
| Python/Make execution | packaged job in a Declarative Automation Bundle |
| local predictor | batch inference or Model Serving at a concrete version |
| simulated drift report | AI Gateway inference table + governed data profiling and delayed labels |

Read [the complete handoff](../docs/databricks-handoff.md) before adapting the
project. Moving to cloud does not authorize infrastructure creation or secrets;
use the approved keyless identity and external platform process.

### Exercise

Design one alert for service health, one for data health, and one for outcome
quality. For each, name an owner and a safe action.

**Hint:** an alert without a response owner and playbook is telemetry, not an
operating control.

**Checkpoint:** you can trace one prediction back to a concrete model
version, threshold, signature, dataset version, training/selection/test runs,
source state, dependency lock, release decision, and monitoring plan.
